In [9]:
from google.colab import files
uploaded = files.upload()

Saving lettuce_dataset_updated.csv to lettuce_dataset_updated.csv


In [10]:
import pandas as pd
df = pd.read_csv('lettuce_dataset_updated.csv', encoding='latin1')
df.head()

,Plant_ID,Date,Temperature (°C),Humidity (%),TDS Value (ppm),pH Level,Growth Days,Temperature (F),Humidity
0,1,8/3/2023,33.4,53,582,6.4,1,92.12,0.53
1,1,8/4/2023,33.5,53,451,6.1,2,92.30,0.53
2,1,8/5/2023,33.4,59,678,6.4,3,92.12,0.59
3,1,8/6/2023,33.4,68,420,6.4,4,92.12,0.68
4,1,8/7/2023,33.4,74,637,6.5,5,92.12,0.74


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    pairwise_distances
)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110

OUTPUT_DIR = "outputKmeans"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("[INFO] Library siap.")

[INFO] Library siap.


In [12]:
df_full = df

In [11]:
candidate_paths = [
    "lettuce_dataset_updated.csv",
    os.path.join("dataset", "lettuce_dataset_updated.csv"),
    os.path.join("/content", "lettuce_dataset_updated.csv"),
]

valid_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if valid_path is None:
    raise FileNotFoundError("File lettuce_dataset_updated.csv tidak ditemukan. Upload dulu ke folder kerja notebook.")

df_full = pd.read_csv(valid_path, encoding='latin1')

In [13]:
FEATURES = ['Temperature (°C)', 'Humidity (%)', 'TDS Value (ppm)', 'pH Level', 'Growth Days']

def preprocess(df):
    # 1. Ambil fitur
    df_features = df[FEATURES].astype(float)

    # 2. Standarisasi (mean 0, std 1)
    X_std = StandardScaler().fit_transform(df_features.values)

    # 3. Normalisasi L2 -> ||x|| = 1
    X_norm = Normalizer(norm='l2').fit_transform(X_std)
    return df_features, X_norm

# Uji pada data full
_, X_test = preprocess(df_full)
print("Shape:", X_test.shape)
print("Norma L2 3 baris pertama (harus 1):", np.linalg.norm(X_test[:3], axis=1))

Shape: (3169, 5)
Norma L2 3 baris pertama (harus 1): [1. 1. 1.]


In [14]:
class KMeansCosine:
    def __init__(self, n_clusters=3, max_iter=300, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.centroids = None
        self.labels_ = None
        self.inertia_ = 0.0
        self.n_iter_ = 0
        self.execution_time_ = 0.0

    def _compute_distance(self, X, centroids):
        return pairwise_distances(X, centroids, metric='cosine')

    def _init_centroids(self, X):
        rng = np.random.RandomState(self.random_state)
        n_samples = X.shape[0]
        centroids = [X[rng.randint(n_samples)]]

        for _ in range(1, self.n_clusters):
            dists = self._compute_distance(X, np.array(centroids))
            sq = np.min(dists, axis=1) ** 2
            probs = np.ones(n_samples) / n_samples if sq.sum() == 0 else sq / sq.sum()
            centroids.append(X[rng.choice(n_samples, p=probs)])

        centroids = np.array(centroids)
        norms = np.linalg.norm(centroids, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        return centroids / norms

    def fit(self, X):
        start = time.perf_counter()

        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10
        X_calc = X / norms

        self.centroids = self._init_centroids(X_calc)
        rng = np.random.RandomState(self.random_state)

        for it in range(self.max_iter):
            self.n_iter_ = it + 1

            # Assignment
            dists = self._compute_distance(X_calc, self.centroids)
            labels = np.argmin(dists, axis=1)

            # Update
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                pts = X_calc[labels == k]
                if len(pts) == 0:
                    new_centroids[k] = X_calc[rng.randint(len(X_calc))]
                else:
                    mean_vec = pts.mean(axis=0)
                    n = np.linalg.norm(mean_vec)
                    new_centroids[k] = mean_vec / (n if n > 0 else 1.0)

            shift = np.linalg.norm(new_centroids - self.centroids)
            self.centroids = new_centroids
            self.labels_ = labels
            if shift < self.tol:
                break

        # Cosine inertia
        final = self._compute_distance(X_calc, self.centroids)
        self.inertia_ = float(final[np.arange(len(X_calc)), self.labels_].sum())
        self.execution_time_ = float(time.perf_counter() - start)
        return self

print("[INFO] Kelas KMeansCosine siap.")

[INFO] Kelas KMeansCosine siap.
